# Step 1
---
* Setup
* Imports
* Data load

In [ ]:
from pathlib import Path
import json
import platform
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import joblib
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score, roc_auc_score, confusion_matrix,
    ConfusionMatrixDisplay, PrecisionRecallDisplay,
)
from sklearn.inspection import permutation_importance

DATA_DIR = "../data/"
ARTIFACT_DIR = "../artifacts/"
dataset_file = DATA_DIR + "ai4i2020.csv"
source = pd.read_csv(dataset_file)

SEED = 42
source.head(2)

# Step2 
---
* Massage dataset and generate synth. data
* Build raw dataset with synthetic data

In [ ]:
RENAME_COLS = {
    "UDI": "udi", "Product ID": "product_id", "Type": "type",
    "Air temperature [K]": "air_temperature",
    "Process temperature [K]": "process_temperature",
    "Rotational speed [rpm]": "rpm", "Torque [Nm]": "torque",
    "Tool wear [min]": "tool_wear", "Machine failure": "machine_failure",
}

def build_ai4i_plus(original, seed=SEED):
    rng = np.random.default_rng(seed)
    raw = original.rename(columns=RENAME_COLS).copy(deep=True)
    n = len(raw)
    raw["machine_id"] = rng.choice([f"M-{i:03d}" for i in range(1, 51)], n)
    raw["batch_id"] = [f"B-{i // 100:04d}" for i in range(n)]
    raw["timestamp"] = pd.date_range("2025-01-01", periods=n, freq="min", tz="UTC")
    raw["shift"] = np.where(raw["timestamp"].dt.hour.between(6, 17), "day", "night")
    raw["machine_age_years"] = rng.uniform(0.5, 15, n)
    raw["maintenance_interval_hours"] = rng.choice([250, 500, 750], n)
    raw["power_consumption_kw"] = (
        raw["torque"] * raw["rpm"] * (2 * np.pi / 60) / 1000
        + rng.normal(0, 0.08, n)
    ).clip(lower=0)
    raw["spindle_load_pct"] = (raw["torque"] / 80 * 100 + rng.normal(0, 2, n)).clip(0, 120)
    raw["vibration_rms"] = (
        0.6 + 0.018 * raw["torque"] + 0.004 * raw["tool_wear"]
        + 0.0006 * (raw["rpm"] - 1500).abs() + rng.normal(0, 0.15, n)
    ).clip(lower=0)
    raw["ambient_humidity"] = rng.uniform(25, 80, n)
    raw["coolant_flow"] = (7 + 0.25 * (raw["process_temperature"] - 310)
                           + rng.normal(0, 0.4, n)).clip(lower=0)
    raw["aux_sensor_noise"] = rng.normal(0, 1, n)
    # Deliberately invalid inputs: recorded after the outcome.
    raw["post_failure_alarm"] = raw["machine_failure"]
    raw["emergency_stop"] = raw["machine_failure"] * rng.binomial(1, 0.9, n)
    modes = ["TWF", "HDF", "PWF", "OSF", "RNF"]
    raw["failure_code"] = raw[modes].apply(
        lambda row: "+".join(row.index[row.eq(1)]) or "NONE", axis=1
    )
    for name in ["air_temperature", "process_temperature", "rpm", "torque",
                 "tool_wear", "vibration_rms", "coolant_flow", "ambient_humidity"]:
        raw.loc[rng.random(n) < 0.02, name] = np.nan
    raw.loc[rng.random(n) < 0.01, "type"] = np.nan
    for name in ["rpm", "vibration_rms"]:
        spikes = rng.choice(n, size=max(1, n // 200), replace=False)
        raw.loc[spikes, name] *= 4
    return raw

raw = build_ai4i_plus(source)
pd.testing.assert_frame_equal(raw, build_ai4i_plus(source))
raw.to_csv(DATA_DIR + "ai4i_plus.csv", index=False)
print("AI4I+ shape:", raw.shape)
display(raw.head())
display(raw.tail())

# Step 3:
---
* Split dataset into Train, test and validate sets - 60, 20, 20 % respectively
* Check that rows are unique amongst the split sets

In [ ]:
train_raw, holdout_raw = train_test_split(
    raw, test_size=0.4, stratify=raw["machine_failure"], random_state=SEED
)
val_raw, test_raw = train_test_split(
    holdout_raw, test_size=0.5, stratify=holdout_raw["machine_failure"], random_state=SEED
)
assert set(train_raw.index).isdisjoint(val_raw.index)
assert set(train_raw.index).isdisjoint(test_raw.index)
assert set(val_raw.index).isdisjoint(test_raw.index)
display(pd.DataFrame({
    "rows": [len(train_raw), len(val_raw), len(test_raw)],
    "failures": [part["machine_failure"].sum() for part in [train_raw, val_raw, test_raw]],
}, index=["train", "validation", "test"]))


missing = train_raw.isna().mean().mul(100).sort_values(ascending=False)
display(missing[missing > 0].to_frame("missing_percent"))
#display(train_raw.describe().T)

# Step 4:
---
* Identify correlations and TARGET columns

## Derived values and formulae

- Mechanical power = torque × angular speed; spindle load tracks torque.
- Vibration increases with torque, wear, and departure from nominal RPM.
- Coolant flow is a fictional controller response to process temperature.
- Humidity, machine age, and auxiliary readings are simulated context/noise.
- Machine/batch IDs and timestamps are artificial and do not turn rows into histories.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
train_raw["machine_failure"].value_counts().sort_index().plot.bar(ax=axes[0])
axes[0].set(title="Training target imbalance", xlabel="Failure label", ylabel="Rows")

axes[1].scatter(train_raw["air_temperature"], train_raw["process_temperature"], s=5, alpha=.25)
axes[1].set(xlabel="Air temperature (K)", ylabel="Process temperature (K)", title="Related temperatures")

axes[2].scatter(train_raw["torque"], train_raw["power_consumption_kw"], s=5, alpha=.25)
axes[2].set(xlabel="Torque (Nm)", ylabel="Generated power (kW)", title="Power also depends on RPM")
plt.tight_layout()
plt.show()

correlations = train_raw.select_dtypes(include="number").corr()["machine_failure"].drop("machine_failure")
display(correlations.reindex(correlations.abs().sort_values(ascending=False).index).to_frame("target_correlation"))

* -1 -> Negative correlation. Lower the value, lower chance of failure
* 0 -> No correlation.
* 1 -> Higher the value, stronger positive correlation

# Step 5
---

* Define input contracts
* Identify leaky and redundant features


In [ ]:
NUMERIC = ["air_temperature", "process_temperature", "rpm", "torque", "tool_wear", "vibration_rms"]
CATEGORICAL = ["type"]
FEATURES = NUMERIC + CATEGORICAL
TARGET = "machine_failure"
LEAKY = ["TWF", "HDF", "PWF", "OSF", "RNF", "post_failure_alarm", "emergency_stop", "failure_code"]
IDENTIFIERS = ["udi", "product_id", "machine_id", "batch_id", "timestamp"]
REDUNDANT = ["power_consumption_kw", "spindle_load_pct", "coolant_flow"]

def feature_decision(name):
    if name == TARGET:
        return "Target", "Known answer used for supervised learning"
    if name in FEATURES:
        return "Keep", "Operating input assumed available at inference"
    if name in LEAKY:
        return "Exclude", "Outcome indicator / post-event information"
    if name in IDENTIFIERS:
        return "Exclude", "Identifier or artificial ordering; not operating evidence"
    if name in REDUNDANT:
        return "Exclude", "Synthetic proxy of retained sensors; keep demo interpretable"
    return "Exclude", "Simulated context/noise with no justified additional mechanism"

decisions = pd.DataFrame(
    [(name, *feature_decision(name)) for name in raw.columns],
    columns=["column", "decision", "reason"]
)
display(decisions)
assert not set(FEATURES).intersection(LEAKY + [TARGET])
X_train, y_train = train_raw[FEATURES], train_raw[TARGET]
X_val, y_val = val_raw[FEATURES], val_raw[TARGET]
raw.head(2)

# Step 6:
---
* Preprocessing
* Impute missing values

In [ ]:
numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median"))])
category_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, NUMERIC),
    ("category", category_pipeline, CATEGORICAL),
], remainder="drop")

def make_pipeline(classifier):
    return Pipeline([("preprocessing", clone(preprocessor)), ("model", classifier)])



# Step 7
---
* Model selection
* Training
* Evaluation

In [ ]:
models = {
    "Dummy": make_pipeline(DummyClassifier(strategy="most_frequent")),
    "Small tree": make_pipeline(DecisionTreeClassifier(
        max_depth=4, min_samples_leaf=20, class_weight="balanced", random_state=SEED)),
    "Deep tree (diagnostic)": make_pipeline(DecisionTreeClassifier(random_state=SEED)),
    "Random Forest": make_pipeline(RandomForestClassifier(
        n_estimators=250, max_depth=12, min_samples_leaf=2,
        class_weight="balanced_subsample", n_jobs=-1, random_state=SEED)),
}

def failure_probability(pipeline, X):
    class_index = list(pipeline.classes_).index(1)
    return pipeline.predict_proba(X)[:, class_index]

def evaluate(y_true, probabilities, threshold=0.5):
    predicted = (probabilities >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y_true, predicted),
        "precision": precision_score(y_true, predicted, zero_division=0),
        "recall": recall_score(y_true, predicted, zero_division=0),
        "f1": f1_score(y_true, predicted, zero_division=0),
        "average_precision": average_precision_score(y_true, probabilities),
        "roc_auc": roc_auc_score(y_true, probabilities),
    }

rows = []
for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    for split, X, y in [("train", X_train, y_train), ("validation", X_val, y_val)]:
        rows.append({"model": name, "split": split, **evaluate(y, failure_probability(pipeline, X))})
comparison = pd.DataFrame(rows).set_index(["model", "split"])
display(comparison.round(4))


1. Accuracy — “How many predictions were correct overall?”
   - Accuracy includes both correctly detected failures and correctly identified normal rows

2. Precision — “When the model raises an alert, how often is it right?”
   - If the model raises 100 alerts and 40 correspond to failures:
     
3. Recall — “Of all the real failures, how many did we catch?”
   - If there are 50 failures and the model detects 40
   - Precision starts with all alerts.
   - Recall starts with all actual failures.
       
4. F1 — “How well are precision and recall balanced?”
   - It combines precision and recall into one number. A low value in either reduces F1 scroe.
      
5. Average precision (AP) — “Does the model generally give failures higher scores?”
   - Higher AP is better. A perfect ranking has AP 1.

In [ ]:
small_tree = models["Small tree"]
names = small_tree.named_steps["preprocessing"].get_feature_names_out()
plt.figure(figsize=(20, 9))
plot_tree(small_tree.named_steps["model"], feature_names=names,
          class_names=["normal", "failure"], filled=True, max_depth=2, fontsize=8)
plt.title("First levels of the small tree (display truncated; model depth is 4)")
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
for name in ["Dummy", "Small tree", "Random Forest"]:
    PrecisionRecallDisplay.from_predictions(
        y_val, failure_probability(models[name], X_val), name=name, ax=ax
    )
ax.axhline(y_val.mean(), color="gray", linestyle="--", label="Validation prevalence")
ax.set_title("Validation precision–recall curves")
ax.legend()
plt.show()

# Step 8
---
* Test and finetune model

In [ ]:
primary = models["Random Forest"]
importance = permutation_importance(
    primary, X_val, y_val, scoring="average_precision", n_repeats=5,
    random_state=SEED, n_jobs=1
)
importance_table = pd.DataFrame({
    "feature": FEATURES, "mean_AP_drop": importance.importances_mean,
    "std_AP_drop": importance.importances_std,
}).sort_values("mean_AP_drop", ascending=False)
display(importance_table)


# Step 9
---
* Freeze thresholds and probabilities
* Cost of False Negative(missed prediction) = 10 Units
* Cost of False Positive(false alarm) = 1 Unit

In [ ]:
FN_COST, FP_COST = 10, 1
val_probabilities = failure_probability(primary, X_val)
#print(val_probabilities)
threshold_rows = []
for threshold in np.linspace(0.01, 0.99, 99):
    predictions = (val_probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, predictions, labels=[0, 1]).ravel()
    threshold_rows.append({
        "threshold": float(threshold), "false_positives": int(fp), "false_negatives": int(fn),
        "cost": int(FP_COST * fp + FN_COST * fn),
        "precision": precision_score(y_val, predictions, zero_division=0),
        "recall": recall_score(y_val, predictions, zero_division=0),
    })
threshold_table = pd.DataFrame(threshold_rows)
selected = threshold_table.sort_values(["cost", "threshold"]).iloc[0]
THRESHOLD = float(selected["threshold"])
display(threshold_table.sort_values(["cost", "threshold"]).head(10))
print(f"Frozen threshold: {THRESHOLD:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
threshold_table.plot(x="threshold", y=["precision", "recall"], ax=axes[0])
threshold_table.plot(x="threshold", y="cost", ax=axes[1])
for ax in axes:
    ax.axvline(THRESHOLD, color="black", linestyle="--")
plt.tight_layout()
plt.show()


# Step 10
---
* Test the model

In [ ]:
X_test, y_test = test_raw[FEATURES], test_raw[TARGET]
test_probabilities = failure_probability(primary, X_test)
test_predictions = (test_probabilities >= THRESHOLD).astype(int)
test_metrics = evaluate(y_test, test_probabilities, THRESHOLD)
display(pd.Series(test_metrics, name="Final test metrics").to_frame())
tn, fp, fn, tp = confusion_matrix(y_test, test_predictions, labels=[0, 1]).ravel()
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")
print(f"Test cost at declared costs: {FP_COST * fp + FN_COST * fn}")
ConfusionMatrixDisplay.from_predictions(
    y_test, test_predictions, labels=[0, 1], display_labels=["normal", "failure"], cmap="Blues"
)
plt.title(f"Final test: threshold={THRESHOLD:.2f}")
plt.show()

errors = X_test.copy()
errors["actual"] = y_test
errors["p_failure"] = test_probabilities
errors["prediction"] = test_predictions
display(errors.loc[errors["actual"] != errors["prediction"]].head(10))

# Step 11
---
* Save the model and metadata to a file


In [ ]:
VERSIONS = {"python": platform.python_version(), "numpy": np.__version__,
            "pandas": pd.__version__, "sklearn": sklearn.__version__,
            "joblib": joblib.__version__}

MODEL_PATH = ARTIFACT_DIR + "machine_failure_pipeline.joblib"
METADATA_PATH = ARTIFACT_DIR + "machine_failure_metadata.json"
joblib.dump(primary, MODEL_PATH)
metadata = {
    "target": TARGET, "features": FEATURES, "numeric_features": NUMERIC,
    "categorical_features": CATEGORICAL, "threshold": THRESHOLD,
    "seed": SEED, "versions": VERSIONS, "test_metrics": test_metrics,
    "false_negative_cost": FN_COST, "false_positive_cost": FP_COST,
    "scope": "Current-state classification on simulated AI4I+; not future forecasting",
}
with open(METADATA_PATH, "w") as fp:
    fp.write(json.dumps(metadata, indent=2))
loaded = joblib.load(MODEL_PATH)
loaded_metadata = json.loads(open(METADATA_PATH).read())
np.testing.assert_allclose(failure_probability(loaded, X_test), test_probabilities)
print("Saved and verified:", MODEL_PATH, "and", METADATA_PATH)


# Step 12
---
* Validate the model with manual input data
  

In [ ]:
BOUNDS = {
    "air_temperature": (250, 400), "process_temperature": (250, 450),
    "rpm": (100, 5000), "torque": (0, 150), "tool_wear": (0, 400),
    "vibration_rms": (0, 20),
}

def predict_event(event, pipeline=loaded, config=loaded_metadata):
    values, notices = {}, []
    for name in config["numeric_features"]:
        value = event.get(name)
        if value is None or (isinstance(value, (int, float)) and np.isnan(value)):
            values[name] = np.nan
            notices.append(f"{name}: missing; imputed")
            continue
        if isinstance(value, bool):
            return {"status": "rejected", "reason": f"{name}: boolean is not a sensor reading"}
        try:
            value = float(value)
        except (TypeError, ValueError):
            return {"status": "rejected", "reason": f"{name}: expected numeric value"}
        low, high = BOUNDS[name]
        if not np.isfinite(value) or not low <= value <= high:
            return {"status": "rejected", "reason": f"{name}: outside demo bounds [{low}, {high}]"}
        values[name] = value
    category = event.get("type")
    if category is None or (isinstance(category, float) and np.isnan(category)):
        category = np.nan
        notices.append("type: missing; imputed")
    elif not isinstance(category, str):
        return {"status": "rejected", "reason": "type: expected a string"}
    elif category not in {"L", "M", "H"}:
        notices.append("type: unknown category; encoded as all zeros")
    values["type"] = category
    if all(pd.isna(values[name]) for name in config["numeric_features"]):
        return {"status": "rejected", "reason": "all numeric sensors are missing"}
    frame = pd.DataFrame([values], columns=config["features"])
    probability = float(failure_probability(pipeline, frame)[0])
    return {"status": "ok", "machine_id": event.get("machine_id"),
            "p_failure": probability, "prediction": int(probability >= config["threshold"]),
            "threshold": config["threshold"], "warnings": notices}

example = {
    "machine_id": "M-DEMO", "type": "L", "air_temperature": 300.0,
    "process_temperature": 310.0, "rpm": 1500, "torque": 40,
    "tool_wear": 100, "vibration_rms": 1.8,
}
display(predict_event(example))
display(predict_event({**example, "torque": None}))
display(predict_event({**example, "type": "UNKNOWN"}))
display(predict_event({**example, "rpm": 100_000}))


##################################
# More validations
##################################

assert predict_event(example)["status"] == "ok"
missing_result = predict_event({k: v for k, v in example.items() if k != "torque"})
assert missing_result["status"] == "ok" and missing_result["warnings"]
assert predict_event({**example, "type": "UNKNOWN"})["warnings"]
assert predict_event({**example, "rpm": 100_000})["status"] == "rejected"
assert predict_event({**example, "torque": "broken"})["status"] == "rejected"
assert predict_event({**example, "torque": float("inf")})["status"] == "rejected"
assert predict_event({})["status"] == "rejected"
assert predict_event({**example, "post_failure_alarm": 1}) == predict_event(example)
medians = primary.named_steps["preprocessing"].named_transformers_["numeric"].named_steps["imputer"].statistics_
np.testing.assert_allclose(medians, X_train[NUMERIC].median().to_numpy())
assert 0 <= predict_event(example)["p_failure"] <= 1
print("All reference checks passed.")



# Step 13
---
* Log metadata and register model with mlflow

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

mlflow.set_tracking_uri("http://localhost:5001")
mlflow.set_experiment("AI4I Machine Failure")

example = X_train.head(5)

with mlflow.start_run(run_name="random-forest") as run:

    mlflow.log_params(primary.named_steps["model"].get_params())
    mlflow.log_param("alert_threshold", float(THRESHOLD))

    mlflow.log_metrics({
        f"test_{name}": float(value)
        for name, value in test_metrics.items()
    })

    
    model_info = mlflow.sklearn.log_model(
        sk_model=primary,
        # Explicitly trust these types in our own locally trained forest.
        serialization_format="skops",
        skops_trusted_types=["numpy.dtype", "sklearn.tree._tree.Tree"],
        name="pipeline",
        registered_model_name="AI4I-Machine-Failure",
        input_example=example,
        signature=infer_signature(
            example, primary.predict_proba(example)
        ),
        pyfunc_predict_fn="predict_proba",
        metadata={
            "alert_threshold": float(THRESHOLD),
            "features": list(X_train.columns),
            "classes": primary.classes_.tolist(),
        },
    )

    print("Run ID:", run.info.run_id)
    print("Model URI:", model_info.model_uri)